# Step 5: Drive recovery
Choose a T4 runtime. This notebook contains the exact locally tested source snapshot. Run the pilot before full extraction. The source and outputs persist on Drive; reconnect, rerun setup, then rerun the interrupted extraction command. Do not edit source during a run.

Local verification: 142 tests, one unavailable-Tesseract skip; actual pretrained image-only GPU recovery passed. Drive/remount and full OCR gates remain pending.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import base64, hashlib, io, os, subprocess, sys, zipfile
STORE = Path('/content/drive/MyDrive/ah-clustering-experiment')
STORE.mkdir(parents=True, exist_ok=True)
for key, relative in {'HF_HOME':'cache/huggingface','HF_HUB_CACHE':'cache/huggingface/hub','HF_DATASETS_CACHE':'cache/datasets','XDG_CACHE_HOME':'cache','PIP_CACHE_DIR':'cache/pip','TMPDIR':'cache/tmp','TMP':'cache/tmp','TEMP':'cache/tmp'}.items():
    directory=STORE/relative
    directory.mkdir(parents=True,exist_ok=True)
    os.environ[key]=str(directory)
os.environ['OMP_THREAD_LIMIT']='1'
SOURCE_DIGEST = '19f95fe14111b8abff529bad79c673e054befb50859b71e68efec76c5e4c63d6'
snapshot = base64.b64decode('...')
assert hashlib.sha256(snapshot).hexdigest() == '425466fb750d78aa201f1ac63f05e86ec04d788f9bae064b0955c3afacc5ff91'
SNAPSHOT = STORE/'source-snapshots'/SOURCE_DIGEST
with zipfile.ZipFile(io.BytesIO(snapshot)) as archive:
    for item in archive.infolist():
        destination=(SNAPSHOT/item.filename).resolve()
        assert destination.is_relative_to(SNAPSHOT.resolve())
        contents=archive.read(item)
        if destination.exists():
            assert destination.read_bytes()==contents, 'Existing source snapshot changed'
        else:
            destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_bytes(contents)
SOURCE=SNAPSHOT/'ah-clustering-experiment'
print('Source:',SOURCE)
print('Storage:',STORE)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(SOURCE/'requirements-embeddings.txt')],check=True)
subprocess.run(['apt-get','update'],check=True)
subprocess.run(['apt-get','install','-y','tesseract-ocr','tesseract-ocr-eng'],check=True)
sys.path.insert(0,str(SOURCE))
from src.provenance import source_identity
assert source_identity(SOURCE)['source_digest']==SOURCE_DIGEST
subprocess.run([sys.executable,str(SOURCE/'experiments/prepare_dataset.py'),'--storage-root',str(STORE),'--split-csv',str(SNAPSHOT/'data/split.csv')],check=True)


In [ ]:
subprocess.run([sys.executable,str(SOURCE/'experiments/verify_recovery_storage.py'),'--storage-root',str(STORE)],check=True)
subprocess.run([sys.executable,str(SOURCE/'experiments/verify_step5_live.py'),'--storage-root',str(STORE),'--model','layoutlmv3-image','--device','cuda:0','--run-id','drive-layoutlmv3-step5'],check=True)


The pilot deliberately terminates a child during a checkpoint write and restarts it. A pass establishes same-runtime process recovery on this mount. Reconnect/remount persistence remains a separate check. Existing complete pilot requests will be reused; repeating the deliberate-crash controller after success is expected to reject the absence of a new crash. Read the saved report instead.

DINOv3 requires previously approved checkpoint access. If access is unavailable, do not substitute weights. Record the failure and run other modes individually.


In [ ]:
import json
ocr_options=STORE/'cache/ocr-options.json'
ocr_options.write_text(json.dumps({'timeout_seconds':600}))
for model in ['dinov3','colpali','colqwen2','layoutlmv3-image','layoutlmv3-ocr']:
    subprocess.run([sys.executable,str(SOURCE/'experiments/extract_embeddings.py'),'--storage-root',str(STORE),'--model',model,'--device','cuda:0','--batch-size','1','--ocr-config',str(ocr_options)],check=True)


After confirming completed manifests and exact IDs on Drive, disconnect and delete the Colab runtime to release compute. Do not treat files only under `/content` as persistent.
